# 01 · Dataset Exploration — DocVQA

**VisionDoc AI** fine-tunes an open-source Vision-Language Model (default
`Qwen/Qwen2.5-VL-3B-Instruct`) with LoRA for *document intelligence*: document
question answering, structured field extraction, confidence scoring, and region
highlighting.

This first notebook is a guided tour of the **DocVQA** corpus — the dataset the
default config trains on. We will:

1. Load the project config (single source of truth for every run).
2. Pull a handful of validation samples through the project's `DocSample` schema.
3. Visualise a few document images alongside their questions and gold answers.
4. Report the split sizes and a quick look at answer-length statistics.

> **Requirements.** This notebook needs the full project dependencies
> (`torch`, `transformers`, `datasets`, `Pillow`, `matplotlib`) and network
> access on first run to download DocVQA into the configured cache. No trained
> adapter is required — exploration only touches raw data.


## Setup

Make the repo importable, then load the config.

In [ ]:
# --- Make the repo root importable (works whether run from notebooks/ or root)
import sys
from pathlib import Path

# Walk up until we find the repo marker (pyproject.toml); fall back to parent.
_here = Path.cwd()
_root = next(
    (p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists()),
    _here.parent,
)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Repo root on sys.path: {_root}")


In [ ]:
# The config is the single source of truth: dataset name, model id, caps, paths.
# load_config() with no argument reads $VISIONDOC_CONFIG or configs/default.yaml.
from configs import load_config
from utils.logging_utils import get_logger

logger = get_logger("nb.dataset_exploration")

config = load_config()  # -> ProjectConfig
print("Project     :", config.name)
print("Dataset     :", config.data.dataset_name, f"({config.data.dataset_id})")
print("Backbone    :", config.model.model_id, f"[{config.model.model_type}]")
print("Image size  :", config.data.image_size)
print("Train cap   :", config.data.max_train_samples)
print("Eval cap    :", config.data.max_eval_samples)


## About DocVQA

**DocVQA** (Document Visual Question Answering) pairs scanned business documents —
invoices, letters, forms, reports, tables — with natural-language questions whose
answers must be *read off the page*. Unlike generic VQA, answers are almost always
short spans of text that physically appear in the image, which makes it an ideal
benchmark for a document-reading VLM:

- The model must jointly perform **layout-aware OCR** and **reading comprehension**.
- The official metric is **ANLS** (Average Normalized Levenshtein Similarity),
  which rewards near-miss transcriptions instead of demanding an exact match —
  well suited to OCR-style answers where a single character can differ.
- The public test split is an *unlabelled challenge set*, so this project carves a
  deterministic (seeded) validation/test partition from the train pool when a
  labelled split is missing — see `preprocessing/datasets.py`.

Every corpus in this project is normalised into a single `DocSample` schema, so
the loaders, trainer, evaluator, and API stay completely dataset-agnostic.


In [ ]:
# load_samples dispatches on config.data.dataset_name and returns a list of the
# project's canonical DocSample objects — never a raw HuggingFace row.
from preprocessing.datasets import load_samples

samples = load_samples(config, "validation", max_samples=8)
print(f"Loaded {len(samples)} validation samples\n")

for s in samples[:8]:
    # A DocSample carries the image (PIL or path), the question, the primary
    # answer, and the full list of acceptable answers (DocVQA ships several).
    print(f"[{s.sample_id or '—':>10}] Q: {s.question}")
    print(f"{'':>13} A: {s.answer!r}  (all: {s.answers})\n")


## A look at the documents

Render a few pages next to their question and gold answer. We route the image through `utils.load_image`, which accepts a PIL image *or* a path string, so this works no matter how the loader materialised the sample.

In [ ]:
import matplotlib.pyplot as plt
from utils.image_utils import load_image

# Show up to four documents in a 2x2 grid.
show = samples[:4]
n = len(show)
cols = 2
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12, 6 * rows))
axes = axes.ravel() if hasattr(axes, "ravel") else [axes]

for ax, sample in zip(axes, show):
    img = load_image(sample.image)  # -> PIL.Image (RGB), from PIL or path
    ax.imshow(img)
    ax.axis("off")
    # Wrap the caption so long questions/answers stay readable above the page.
    q = sample.question if len(sample.question) < 70 else sample.question[:67] + "..."
    ax.set_title(f"Q: {q}\nA: {sample.answer}", fontsize=10, loc="left")

# Hide any unused axes (when fewer than rows*cols samples are available).
for ax in axes[n:]:
    ax.axis("off")

fig.suptitle("DocVQA — sample pages, questions, and gold answers", fontsize=13)
fig.tight_layout()
plt.show()


## Split sizes

`build_splits` loads all three canonical splits (`train`/`validation`/`test`), applying the sample caps from the config. On a first run this triggers the DocVQA download, so it can take a while; the caps in `default.yaml` keep a smoke run tractable.

In [ ]:
# NOTE: this materialises every split (subject to config caps) and will download
# DocVQA on a cold cache. Comment this cell out for a purely offline peek.
from preprocessing.datasets import build_splits

splits = build_splits(config)
for name, split_samples in splits.items():
    print(f"{name:>11}: {len(split_samples):>6,} samples")


In [ ]:
# Quick distribution of answer lengths (in characters) on the validation split.
# Short answers dominate DocVQA — most are a single value read off the page.
val = splits["validation"]
lengths = [len(s.answer) for s in val if s.answer]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lengths, bins=30, color="#3b82f6", edgecolor="white")
ax.set_title("DocVQA validation — answer length distribution")
ax.set_xlabel("answer length (characters)")
ax.set_ylabel("count")
fig.tight_layout()
plt.show()

if lengths:
    mean_len = sum(lengths) / len(lengths)
    print(f"answers: {len(lengths):,}  |  mean length: {mean_len:.1f} chars"
          f"  |  max: {max(lengths)} chars")


## Takeaways

- DocVQA answers are short, page-grounded spans — a natural fit for a
  document-reading VLM scored with **ANLS**.
- The `DocSample` schema is the contract every downstream module relies on;
  swapping to CORD / FUNSD / SROIE is a one-line `data.dataset_name` change.

**Next:** `02_inference_demo.ipynb` runs the model end-to-end on one of these
pages — answering a question, extracting structured fields, and highlighting the
region the answer came from.
